In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import Window

### **Data Reading**

In [0]:
df = spark.read.format('parquet')\
    .load("abfss://bronze@stretaildevwestindia.dfs.core.windows.net/orders")

In [0]:
display(df)

In [0]:

df=df.drop("_rescued_data")
df.display()

In [0]:

df = df.withColumn("order_date", to_timestamp(col("order_date")))
df.display()

In [0]:
df = df.withColumn("year", year(col('order_date')))
df.display()

In [0]:
df1 = df.withColumn("flag", dense_rank().over(Window.partitionBy(col('year')).orderBy(desc('total_amount'))))
df1.display()

### Making window functions easily reusable through classes.

In [0]:


class windows:
    @staticmethod
    def _window(partition_col, order_col, ascending=True):
        order_expr = col(order_col).asc() if ascending else col(order_col).desc()
        return Window.partitionBy(partition_col).orderBy(order_expr)

    @staticmethod
    def add_row_number(df, partition_col, order_col, ascending=True, out_col="row_number"):
        w = windows._window(partition_col, order_col, ascending)
        return df.withColumn(out_col, row_number().over(w))

    @staticmethod
    def add_rank(df, partition_col, order_col, ascending=True, out_col="rank"):
        w = windows._window(partition_col, order_col, ascending)
        return df.withColumn(out_col, rank().over(w))

    @staticmethod
    def add_dense_rank(df, partition_col, order_col, ascending=True, out_col="dense_rank"):
        w = windows._window(partition_col, order_col, ascending)
        return df.withColumn(out_col, dense_rank().over(w))


In [0]:
obj = windows()
df_new = df

In [0]:
df_result = windows().add_rank(df_new,'year','total_amount',ascending=False, out_col="DR")
df_result.display()

## **Data Writing** /(df dataframe no ranking/)

In [0]:
df.write.format("delta").mode("overwrite").save("abfss://silver@stretaildevwestindia.dfs.core.windows.net/orders")

In [0]:
%sql
CREATE TABLE IF NOT EXISTS retail.silver.orders_silver
USING DELTA
LOCATION "abfss://silver@stretaildevwestindia.dfs.core.windows.net/orders"